# Evidence Review

Human review notebook for the Financial Privacy Gateway evidence dataset. This notebook displays `output/evidence.json` for review only. It does not create new Evidence, does not change `review_status`, and does not create `policy_rules.json`.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Markdown

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
evidence_path = ROOT / 'output' / 'evidence.json'
records = json.loads(evidence_path.read_text(encoding='utf-8'))

pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', 0)

df = pd.DataFrame(records)
df['evidence_num'] = df['evidence_id'].str.replace('EV-', '', regex=False).astype(int)

coverage_ids = {
    '개인정보/가명정보': ['EV-00001','EV-00002','EV-00003','EV-00004','EV-00005','EV-00006','EV-00007','EV-00008','EV-00009','EV-00010','EV-00014','EV-00015','EV-00016','EV-00017','EV-00018','EV-00019','EV-00020','EV-00021','EV-00022','EV-00023','EV-00024','EV-00038','EV-00040','EV-00044','EV-00046'],
    '개인신용정보': ['EV-00011','EV-00012','EV-00013','EV-00038','EV-00044','EV-00046'],
    '제3자 제공/위탁': ['EV-00002','EV-00014','EV-00015','EV-00016','EV-00017','EV-00018','EV-00019','EV-00020','EV-00037','EV-00045'],
    '국외이전': ['EV-00021','EV-00022','EV-00023','EV-00024','EV-00041'],
    '금융 AI 이용': ['EV-00035','EV-00036','EV-00037','EV-00038','EV-00039','EV-00040','EV-00041','EV-00042'],
    'SaaS/Cloud 이용': ['EV-00028','EV-00029','EV-00038','EV-00039','EV-00041','EV-00043','EV-00044','EV-00045','EV-00046','EV-00047','EV-00048','EV-00049'],
    '암호화/접근통제/로그': ['EV-00016','EV-00025','EV-00026','EV-00027','EV-00030','EV-00031','EV-00032','EV-00033','EV-00034','EV-00038','EV-00039','EV-00040','EV-00045','EV-00047','EV-00048','EV-00049'],
    '재식별/추가정보 관리': ['EV-00006','EV-00007','EV-00008','EV-00009','EV-00010','EV-00040'],
}
evidence_to_coverage = {}
for area, ids in coverage_ids.items():
    for evidence_id in ids:
        evidence_to_coverage.setdefault(evidence_id, []).append(area)

df['coverage_area'] = df['evidence_id'].map(lambda evidence_id: ', '.join(evidence_to_coverage.get(evidence_id, ['UNMAPPED'])))

review_columns = [
    'evidence_id',
    'coverage_area',
    'source_name',
    'article',
    'paragraph',
    'original_text',
    'applies_to',
    'data_type',
    'condition',
    'required_action',
    'prohibition',
    'exception',
    'review_status',
    'source_url',
]

review_df = df.sort_values('evidence_num')[review_columns].fillna('')
display(Markdown(f'Loaded **{len(review_df)}** Evidence records from `{evidence_path}`.'))

## 전체 Evidence 검토 테이블

`original_text`와 구조화 필드를 같은 행에서 비교한다. `coverage_area`는 검토 편의를 위한 노트북 파생 컬럼이며 Evidence 원본 데이터에는 쓰지 않는다.

In [ ]:
display(review_df)

## Review Status 필터

`status_filter` 값을 `PENDING`, `REVIEW_REQUIRED`, `CONFIRMED` 중 하나로 바꿔서 실행한다.

In [ ]:
status_filter = 'PENDING'
filtered_df = review_df[review_df['review_status'] == status_filter]
display(Markdown(f'### {status_filter}: {len(filtered_df)} records'))
display(filtered_df)

In [ ]:
status_filter = 'REVIEW_REQUIRED'
filtered_df = review_df[review_df['review_status'] == status_filter]
display(Markdown(f'### {status_filter}: {len(filtered_df)} records'))
display(filtered_df)

In [ ]:
status_filter = 'CONFIRMED'
filtered_df = review_df[review_df['review_status'] == status_filter]
display(Markdown(f'### {status_filter}: {len(filtered_df)} records'))
display(filtered_df)

## Evidence 단건 검토

`evidence_id_to_review` 값을 바꿔 실행한다. `original_text`는 Markdown 코드블록으로 출력해 잘리지 않게 표시한다.

In [ ]:
evidence_id_to_review = 'EV-00001'
row = review_df.loc[review_df['evidence_id'] == evidence_id_to_review]

if row.empty:
    display(Markdown(f'No Evidence found for `{evidence_id_to_review}`.'))
else:
    item = row.iloc[0].to_dict()
    display(Markdown(f"## {item['evidence_id']}"))
    display(Markdown(
        f"**coverage_area:** {item['coverage_area']}  \n"
        f"**source_name:** {item['source_name']}  \n"
        f"**article:** {item['article']}  \n"
        f"**paragraph:** {item['paragraph']}  \n"
        f"**review_status:** {item['review_status']}  \n"
        f"**source_url:** {item['source_url']}"
    ))
    display(Markdown('### original_text'))
    display(Markdown('```text\n' + str(item['original_text']) + '\n```'))
    display(Markdown('### structured fields'))
    structured = pd.DataFrame([
        {'field': 'applies_to', 'value': item['applies_to']},
        {'field': 'data_type', 'value': item['data_type']},
        {'field': 'condition', 'value': item['condition']},
        {'field': 'required_action', 'value': item['required_action']},
        {'field': 'prohibition', 'value': item['prohibition']},
        {'field': 'exception', 'value': item['exception']},
    ])
    display(structured)

## 요약

In [ ]:
summary = pd.DataFrame([
    {'metric': '전체 Evidence 수', 'count': len(review_df)},
    {'metric': 'PENDING 수', 'count': int((review_df['review_status'] == 'PENDING').sum())},
    {'metric': 'REVIEW_REQUIRED 수', 'count': int((review_df['review_status'] == 'REVIEW_REQUIRED').sum())},
    {'metric': 'CONFIRMED 수', 'count': int((review_df['review_status'] == 'CONFIRMED').sum())},
])
display(summary)

In [1]:
import pandas as pd
import json

with open("../output/evidence.json", "r", encoding="utf-8") as f:
    evidence = json.load(f)

df = pd.DataFrame(evidence)

cols = [
    "evidence_id",
    "coverage_area",
    "source_name",
    "article",
    "paragraph",
    "original_text",
    "applies_to",
    "data_type",
    "condition",
    "required_action",
    "prohibition",
    "exception",
    "review_status",
    "source_url"
]

cols = [c for c in cols if c in df.columns]

print(df[cols].to_json(
    orient="records",
    force_ascii=False,
    indent=2
))

[
  {
    "evidence_id":"EV-00001",
    "source_name":"개인정보 보호법",
    "article":"제28조의2",
    "paragraph":"①",
    "original_text":"개인정보처리자는 통계작성, 과학적 연구, 공익적 기록보존 등을 위하여 정보주체의 동의 없이 가명정보를 처리할 수 있다.",
    "applies_to":"개인정보처리자",
    "data_type":"가명정보",
    "condition":"통계작성, 과학적 연구, 공익적 기록보존",
    "required_action":null,
    "prohibition":null,
    "exception":"동의 없이",
    "review_status":"PENDING",
    "source_url":"https:\/\/law.go.kr\/LSW\/lsInfoP.do?lsiSeq=270351"
  },
  {
    "evidence_id":"EV-00002",
    "source_name":"개인정보 보호법",
    "article":"제28조의2",
    "paragraph":"②",
    "original_text":"개인정보처리자는 제1항에 따라 가명정보를 제3자에게 제공하는 경우에는 특정 개인을 알아보기 위하여 사용될 수 있는 정보를 포함해서는 아니 된다.",
    "applies_to":"개인정보처리자",
    "data_type":"가명정보",
    "condition":"제3자에게 제공하는 경우",
    "required_action":null,
    "prohibition":"포함해서는 아니 된다",
    "exception":null,
    "review_status":"PENDING",
    "source_url":"https:\/\/law.go.kr\/LSW\/lsInfoP.do?lsiSeq=270351"
  },
  {
    "evidence_id":"EV-00003",
 

In [2]:
df[cols].to_json(
    "evidence_review_49.json",
    orient="records",
    force_ascii=False,
    indent=2
)

print("저장 완료: evidence_review_49.json")

저장 완료: evidence_review_49.json


In [3]:
import pandas as pd
import json

with open("../output/evidence.json", "r", encoding="utf-8") as f:
    evidence = json.load(f)

df = pd.DataFrame(evidence)

df.to_json(
    "evidence_review_49_FIXED.json",
    orient="records",
    force_ascii=False,
    indent=2
)

print("완료")

완료
